# ControlNet Fine-tuning — Clothing Sketch + Color Hints

**Architecture:** SD v1.5 + ControlNet lineart (7-channel condition) + UNet LoRA

**Condition channels:**
- Ch 0-2: Sketch (grayscale × 3)
- Ch 3-5: Color hint map (RGB)
- Ch 6:   Hint mask (binary — 1 where user drew color)

**Checkpoint strategy:** Save every 2000 steps. Visual samples every 500 steps.

**Resume:** Automatically resumes from latest checkpoint if found.

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────
import subprocess
pkgs = [
    '-U',
    'diffusers',
    'transformers',
    'accelerate',
    'peft',
    'huggingface_hub',
    'safetensors',
    'bitsandbytes',
    'xformers',
    'scipy',
    'Pillow',
]
subprocess.run(['pip', 'install', '-q'] + pkgs, check=True)
print('All packages installed. Restart the runtime after this cell if packages were upgraded.')

In [ ]:
import os, json, csv, random, glob, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.ndimage import (
    gaussian_filter, binary_dilation, map_coordinates, label as scipy_label
)

from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
    AutoencoderKL,
    DDPMScheduler,
    UNet2DConditionModel,
)
from diffusers.optimization import get_cosine_schedule_with_warmup
from transformers import CLIPTextModel, CLIPTokenizer, CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

# ── CONFIG ─────────────────────────────────────────────────────────────────
CONFIG = {
    # paths
    'train_sketch_dir':  '/kaggle/input/datasets/aimensajjad/data-scribbler/sketches/sketches/train',
    'train_gt_dir':      '/kaggle/input/datasets/aimensajjad/data-scribbler/clothes/clothes/train',
    'val_sketch_dir':    '/kaggle/input/datasets/aimensajjad/data-scribbler/sketches/sketches/val',
    'val_gt_dir':        '/kaggle/input/datasets/aimensajjad/data-scribbler/clothes/clothes/val',
    'output_dir':        '/kaggle/working/controlnet_clothing',
    'samples_dir':       '/kaggle/working/controlnet_samples',
    'logs_dir':          '/kaggle/working/controlnet_logs',

    # base models
    'sd_model':          'runwayml/stable-diffusion-v1-5',
    'controlnet_base':   'lllyasviel/control_v11p_sd15_lineart',

    # training
    'image_size':        512,
    'batch_size':        1,
    'grad_accum_steps':  8,       # effective batch = 8, fits 15GB GPUs
    'lr_controlnet':     1e-5,
    'lr_lora':           5e-5,    # LoRA slightly higher LR
    'use_8bit_adam':     True,    # avoids first optimizer-step OOM on 15GB GPUs
    'max_train_steps':   30000,
    'warmup_steps':      500,
    'save_every_steps':  2000,    # checkpoint frequency
    'sample_every_steps':500,     # visual sample frequency
    'seed':              42,

    # LoRA
    'lora_rank':         4,
    'lora_alpha':        4,

    # hint generation
    'mode_probs':           [0.2, 0.5, 0.3],  # no_hint, sparse, propagated
    'max_strokes_range':    (5, 25),
    'stroke_radius_range':  (4, 12),
    'n_propagated_regions': (2, 6),

    # CLIP category labels
    'categories': [
        't-shirt', 'shirt', 'blouse', 'sweater', 'hoodie',
        'jacket', 'coat', 'dress', 'skirt', 'trousers', 'shorts'
    ],
    'clip_confidence_threshold': 0.55,
    'fallback_prompt': 'a clothing garment, white background, studio product photo, high quality',

    # sketch augmentation
    'aug_prob': 0.6,   # probability of applying augmentation per sample
}

for d in [CONFIG['output_dir'], CONFIG['samples_dir'], CONFIG['logs_dir']]:
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## CLIP Category Labeler

In [ ]:
class CLIPCategoryLabeler:
    """
    Zero-shot CLIP classification to auto-label garment categories.
    Caches results to disk so classification only runs once.
    """
    def __init__(self, categories, cache_path, confidence_threshold=0.55,
                 fallback_prompt='a clothing garment, white background, studio product photo'):
        self.categories           = categories
        self.cache_path           = cache_path
        self.confidence_threshold = confidence_threshold
        self.fallback_prompt      = fallback_prompt
        self.cache                = {}

        # Load cache if exists
        if os.path.exists(cache_path):
            with open(cache_path) as f:
                self.cache = json.load(f)
            print(f'CLIP label cache loaded: {len(self.cache)} entries')
        else:
            print('No CLIP cache found — will classify on first access.')
            self._load_clip()

    def _load_clip(self):
        print('Loading CLIP model for classification...')
        self.clip_model     = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device)
        self.clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        self.clip_model.eval()
        # Pre-compute text embeddings for all categories
        self.text_prompts = [f'a photo of a {c}' for c in self.categories]
        print(f'CLIP ready. Categories: {self.categories}')

    def get_prompt(self, image_path):
        key = str(image_path)
        if key in self.cache:
            return self.cache[key]

        # Classify
        if not hasattr(self, 'clip_model'):
            self._load_clip()

        img = Image.open(image_path).convert('RGB')
        inputs = self.clip_processor(
            text=self.text_prompts, images=img,
            return_tensors='pt', padding=True
        ).to(device)

        with torch.no_grad():
            outputs = self.clip_model(**inputs)
            probs   = outputs.logits_per_image.softmax(dim=-1)[0]

        top_prob, top_idx = probs.max(0)
        if top_prob.item() >= self.confidence_threshold:
            category = self.categories[top_idx.item()]
            prompt   = f'a {category}, white background, studio product photo, high quality'
        else:
            prompt = self.fallback_prompt

        self.cache[key] = prompt
        return prompt

    def save_cache(self):
        with open(self.cache_path, 'w') as f:
            json.dump(self.cache, f)

    def build_full_cache(self, image_paths, save_every=1000):
        """Pre-classify all images and cache results."""
        print(f'Building CLIP label cache for {len(image_paths)} images...')
        for i, p in enumerate(tqdm(image_paths)):
            self.get_prompt(p)
            if (i + 1) % save_every == 0:
                self.save_cache()
                print(f'  Cache saved at {i+1} images')
        self.save_cache()
        print(f'Cache complete: {len(self.cache)} entries')


print('CLIPCategoryLabeler ready.')

## Sketch Augmentation

In [ ]:
def augment_sketch_to_handdrawn(sketch_np, seed=None):
    """
    Simulates hand-drawn sketch quality on clean edge maps.
    Strength is gated by line darkness so faint sketches are protected.

    sketch_np: (H, W) float32 in [0, 1] — 0=black line, 1=white bg
    Returns:   (H, W) float32 in [0, 1]
    """
    if seed is not None:
        np.random.seed(seed)

    out = sketch_np.copy()

    # Measure line darkness — how strong are the lines
    line_strength = 1.0 - sketch_np.mean()

    # Scale augmentation to line strength
    # < 0.01 → truly blank/invisible → intensity 0 (skip everything)
    # > 0.06 → dark clear lines    → intensity 1 (full augmentation)
    intensity = np.clip((line_strength - 0.01) / 0.05, 0.0, 1.0)

    # ── 1. Elastic distortion — wobbly lines ──────────────────────────────
    if np.random.rand() > 0.3:
        sigma    = np.random.uniform(6, 10)
        strength = np.random.uniform(1.5, 3.5) * max(intensity, 0.3)
        if strength > 0.1:
            shape = out.shape
            dx = gaussian_filter(np.random.randn(*shape), sigma=sigma) * strength
            dy = gaussian_filter(np.random.randn(*shape), sigma=sigma) * strength
            x, y = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
            indices = (
                np.clip((y + dy).flatten(), 0, shape[0]-1),
                np.clip((x + dx).flatten(), 0, shape[1]-1)
            )
            out = map_coordinates(out, indices, order=1).reshape(shape)

    # ── 2. Partial thickness variation — only some lines get thicker ──────
    # Applied in a random region so not all lines change uniformly
    if np.random.rand() > 0.4 and intensity > 0.05:
        h, w = out.shape
        # Random region covering 30-70% of image
        r1 = np.random.randint(0, int(h * 0.4))
        r2 = np.random.randint(int(h * 0.6), h)
        c1 = np.random.randint(0, int(w * 0.4))
        c2 = np.random.randint(int(w * 0.6), w)
        region_mask = np.zeros_like(out, dtype=bool)
        region_mask[r1:r2, c1:c2] = True
        binary  = out < 0.5
        dilated = binary_dilation(binary, iterations=1)
        # Only thicken lines inside the selected region
        out = np.where(region_mask & dilated, 0.0, out).astype(np.float32)

    # ── 3. Slight blur — soft edges ───────────────────────────────────────
    # No gaps — dataset already has natural incompleteness from light items
    if np.random.rand() > 0.2:
        sigma = np.random.uniform(0.2, 0.4 + 0.5 * intensity)
        out   = gaussian_filter(out, sigma=sigma).astype(np.float32)

    return np.clip(out, 0, 1).astype(np.float32)


print('Sketch augmentation ready.')

## Hint Generation (Mixed Training)

In [ ]:
def generate_no_hint(img_size):
    color_map = np.zeros((img_size, img_size, 3), dtype=np.float32)
    hint_mask = np.zeros((img_size, img_size, 1), dtype=np.float32)
    return color_map, hint_mask


def generate_sparse_hints(gt_image, max_strokes, stroke_radius, img_size):
    color_map = np.zeros((img_size, img_size, 3), dtype=np.float32)
    hint_mask = np.zeros((img_size, img_size, 1), dtype=np.float32)
    non_bg    = np.argwhere(np.any(gt_image < 0.75, axis=-1))
    n_strokes = np.random.randint(1, max_strokes + 1)
    if len(non_bg) < 10:
        ys = np.random.randint(0, img_size, n_strokes)
        xs = np.random.randint(0, img_size, n_strokes)
    else:
        chosen = non_bg[np.random.choice(len(non_bg), n_strokes)]
        ys, xs = chosen[:, 0], chosen[:, 1]
    yy, xx = np.ogrid[:img_size, :img_size]
    for y, x in zip(ys, xs):
        mask = (xx - x)**2 + (yy - y)**2 <= stroke_radius**2
        color_map[mask]    = gt_image[y, x]
        hint_mask[mask, 0] = 1.0
    return color_map, hint_mask


def generate_propagated_hints(gt_image, sketch_np, img_size, n_seed_regions=4):
    color_map   = np.zeros((img_size, img_size, 3), dtype=np.float32)
    hint_mask   = np.zeros((img_size, img_size, 1), dtype=np.float32)
    sketch_gray = sketch_np[:, :, 0] if sketch_np.ndim == 3 else sketch_np
    edge_binary = sketch_gray < 0.5
    non_edge    = (~edge_binary).astype(np.uint8)
    labeled, n_regions = scipy_label(non_edge)
    if n_regions == 0:
        return color_map, hint_mask
    border_ids = set(
        labeled[0,:].tolist() + labeled[-1,:].tolist() +
        labeled[:,0].tolist() + labeled[:,-1].tolist()
    )
    valid_ids = [i for i in range(1, n_regions + 1) if i not in border_ids]
    if not valid_ids:
        return color_map, hint_mask
    n_to_fill  = min(n_seed_regions, len(valid_ids))
    chosen_ids = np.random.choice(valid_ids, n_to_fill, replace=False)
    for region_id in chosen_ids:
        region_mask   = labeled == region_id
        region_pixels = gt_image[region_mask]
        if len(region_pixels) < 4:
            continue
        dominant_color           = np.median(region_pixels, axis=0)
        color_map[region_mask]   = dominant_color
        hint_mask[region_mask,0] = 1.0
    return color_map, hint_mask


print('Hint generation ready.')

## Dataset

In [ ]:
class ClothingControlNetDataset(Dataset):
    """
    Returns 7-channel condition:
        Ch 0-2: sketch (grayscale × 3)
        Ch 3-5: color hint map (RGB)
        Ch 6:   hint mask (binary)
    """
    def __init__(self, sketch_dir, gt_dir, tokenizer,
                 labeler=None,
                 fallback_prompt='a clothing garment, white background, studio product photo, high quality',
                 img_size=512,
                 mode_probs=(0.2, 0.5, 0.3),
                 max_strokes_range=(5, 25),
                 stroke_radius_range=(4, 12),
                 n_propagated_regions=(2, 6),
                 aug_prob=0.6,
                 augment=False):

        self.sketch_paths = sorted(Path(sketch_dir).glob('*.*'))
        self.gt_paths     = sorted(Path(gt_dir).glob('*.*'))
        assert len(self.sketch_paths) == len(self.gt_paths), \
            f'Mismatch: {len(self.sketch_paths)} sketches vs {len(self.gt_paths)} gt'

        self.tokenizer            = tokenizer
        self.labeler              = labeler
        self.fallback_prompt      = fallback_prompt
        self.img_size             = img_size
        self.mode_probs           = list(mode_probs)
        self.max_strokes_range    = max_strokes_range
        self.stroke_radius_range  = stroke_radius_range
        self.n_propagated_regions = n_propagated_regions
        self.aug_prob             = aug_prob
        self.augment              = augment

        # Pre-tokenize fallback prompt
        self.fallback_ids = self._tokenize(fallback_prompt)

    def _tokenize(self, prompt):
        return self.tokenizer(
            prompt,
            padding='max_length',
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids[0]

    def __len__(self):
        return len(self.sketch_paths)

    def __getitem__(self, idx):
        # ── Load sketch ──
        sketch_pil = Image.open(self.sketch_paths[idx]).convert('L')
        sketch_pil = sketch_pil.resize((self.img_size, self.img_size), Image.BILINEAR)
        sketch_np  = np.array(sketch_pil, dtype=np.float32) / 255.0   # (H,W)

        # ── Load GT ──
        gt_pil = Image.open(self.gt_paths[idx]).convert('RGB')
        gt_pil = gt_pil.resize((self.img_size, self.img_size), Image.BILINEAR)
        gt_np  = np.array(gt_pil, dtype=np.float32) / 255.0           # (H,W,3)

        # ── Flip augmentation ──
        if self.augment and np.random.rand() > 0.5:
            sketch_np = np.fliplr(sketch_np).copy()
            gt_np     = np.fliplr(gt_np).copy()

        # ── Sketch augmentation (simulate hand-drawn) ──
        if self.augment and np.random.rand() < self.aug_prob:
            sketch_np = augment_sketch_to_handdrawn(sketch_np)

        # ── Repeat sketch to 3ch ──
        sketch_3ch = np.stack([sketch_np]*3, axis=-1)                  # (H,W,3)

        # ── Hint generation ──
        max_strokes   = np.random.randint(*self.max_strokes_range)
        stroke_radius = np.random.randint(*self.stroke_radius_range)
        n_regions     = np.random.randint(*self.n_propagated_regions)
        mode = np.random.choice(
            ['no_hint', 'sparse', 'propagated'], p=self.mode_probs
        )

        if mode == 'no_hint':
            color_map, hint_mask = generate_no_hint(self.img_size)
        elif mode == 'sparse':
            color_map, hint_mask = generate_sparse_hints(
                gt_np, max_strokes, stroke_radius, self.img_size)
        else:
            color_map, hint_mask = generate_propagated_hints(
                gt_np, sketch_3ch, self.img_size, n_regions)

        # ── Build 7-channel condition ──
        # Ch 0-2: sketch, Ch 3-5: color hint, Ch 6: hint mask
        condition = np.concatenate(
            [sketch_3ch, color_map, hint_mask], axis=-1
        )  # (H,W,7)

        # ── Normalize to [-1, 1] ──
        condition = condition * 2.0 - 1.0
        gt_np     = gt_np     * 2.0 - 1.0

        # ── To tensor ──
        condition_t  = torch.from_numpy(condition).permute(2,0,1).float()  # (7,H,W)
        pixel_values = torch.from_numpy(gt_np).permute(2,0,1).float()      # (3,H,W)

        # ── Prompt ──
        if self.labeler is not None:
            prompt    = self.labeler.get_prompt(self.gt_paths[idx])
            input_ids = self._tokenize(prompt)
        else:
            input_ids = self.fallback_ids

        return {
            'condition':    condition_t,
            'pixel_values': pixel_values,
            'input_ids':    input_ids,
        }


print('Dataset ready.')

## Build CLIP Label Cache
Run this cell once before training. It classifies all GT images and saves results to disk.
On resume it loads from cache instantly.

In [ ]:
cache_path = os.path.join(CONFIG['logs_dir'], 'clip_label_cache.json')

labeler = CLIPCategoryLabeler(
    categories            = CONFIG['categories'],
    cache_path            = cache_path,
    confidence_threshold  = CONFIG['clip_confidence_threshold'],
    fallback_prompt       = CONFIG['fallback_prompt'],
)

# Build cache if not already done
if len(labeler.cache) == 0:
    all_gt_paths = sorted(Path(CONFIG['train_gt_dir']).glob('*.*'))
    labeler.build_full_cache(all_gt_paths)
    # Unload CLIP to free VRAM before training
    del labeler.clip_model, labeler.clip_processor
    torch.cuda.empty_cache()
    print('CLIP model unloaded — VRAM freed for training.')
else:
    print(f'Using existing cache: {len(labeler.cache)} labels')

# Inspect label distribution
from collections import Counter
label_counts = Counter()
for prompt in labeler.cache.values():
    # Extract category from prompt
    for cat in CONFIG['categories']:
        if cat in prompt:
            label_counts[cat] += 1
            break
    else:
        label_counts['fallback'] += 1

print('\nLabel distribution:')
for cat, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat:<20} {count:>6}')

## Load Models

In [ ]:
print('Loading models...')

tokenizer = CLIPTokenizer.from_pretrained(
    CONFIG['sd_model'], subfolder='tokenizer')

text_encoder = CLIPTextModel.from_pretrained(
    CONFIG['sd_model'], subfolder='text_encoder')
text_encoder.requires_grad_(False)
text_encoder.to(device)

vae = AutoencoderKL.from_pretrained(
    CONFIG['sd_model'], subfolder='vae')
vae.requires_grad_(False)
vae.to(device)

noise_scheduler = DDPMScheduler.from_pretrained(
    CONFIG['sd_model'], subfolder='scheduler')

# ── UNet with LoRA ─────────────────────────────────────────────────────────
unet = UNet2DConditionModel.from_pretrained(
    CONFIG['sd_model'], subfolder='unet')
unet.requires_grad_(False)   # freeze base weights

lora_config = LoraConfig(
    r              = CONFIG['lora_rank'],
    lora_alpha     = CONFIG['lora_alpha'],
    target_modules = ['to_q', 'to_v'],   # attention projections only
    lora_dropout   = 0.0,
)
unet = get_peft_model(unet, lora_config)
unet.to(device)
unet.train()

lora_params = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print(f'UNet LoRA trainable params: {lora_params/1e6:.2f}M')

# ── ControlNet with 7-channel condition input ──────────────────────────────
controlnet = ControlNetModel.from_pretrained(CONFIG['controlnet_base'])

# Modify CONDITION input conv: 3ch → 7ch.
# Do not modify controlnet.conv_in; that is the 4-channel latent conv.
cond_embed = controlnet.controlnet_cond_embedding
old_conv = cond_embed.conv_in
new_conv = nn.Conv2d(
    in_channels  = 7,
    out_channels = old_conv.out_channels,
    kernel_size  = old_conv.kernel_size,
    stride       = old_conv.stride,
    padding      = old_conv.padding,
    dilation     = old_conv.dilation,
    groups       = old_conv.groups,
    bias         = old_conv.bias is not None,
    padding_mode = old_conv.padding_mode,
)
with torch.no_grad():
    # Ch 0-2 (sketch): copy pretrained lineart/RGB condition weights
    new_conv.weight[:, :3, :, :] = old_conv.weight[:, :3, :, :].clone()
    # Ch 3-5 (color hint): zero init — learned from scratch
    new_conv.weight[:, 3:6, :, :] = 0.0
    # Ch 6 (hint mask): zero init
    new_conv.weight[:, 6:7, :, :] = 0.0
    if old_conv.bias is not None:
        new_conv.bias.copy_(old_conv.bias)

cond_embed.conv_in = new_conv
controlnet.config.conditioning_channels = 7
controlnet.train()
controlnet.to(device)

# Gradient checkpointing is needed for 512px ControlNet + UNet LoRA on 15GB GPUs.
if hasattr(controlnet, 'enable_gradient_checkpointing'):
    controlnet.enable_gradient_checkpointing()
if hasattr(unet, 'enable_gradient_checkpointing'):
    unet.enable_gradient_checkpointing()

if hasattr(controlnet, 'enable_xformers_memory_efficient_attention'):
    try:
        controlnet.enable_xformers_memory_efficient_attention()
        unet.enable_xformers_memory_efficient_attention()
        print('xformers enabled.')
    except Exception as e:
        print(f'xformers not available: {e}')

cn_params = sum(p.numel() for p in controlnet.parameters() if p.requires_grad)
print(f'ControlNet trainable params: {cn_params/1e6:.1f}M')
print(f'Input conv: 3ch → 7ch (sketch + color + mask)')
print(f'Models loaded.')

## Dataloaders

In [ ]:
train_dataset = ClothingControlNetDataset(
    CONFIG['train_sketch_dir'], CONFIG['train_gt_dir'],
    tokenizer,
    labeler              = labeler,
    fallback_prompt      = CONFIG['fallback_prompt'],
    img_size             = CONFIG['image_size'],
    mode_probs           = CONFIG['mode_probs'],
    max_strokes_range    = CONFIG['max_strokes_range'],
    stroke_radius_range  = CONFIG['stroke_radius_range'],
    n_propagated_regions = CONFIG['n_propagated_regions'],
    aug_prob             = CONFIG['aug_prob'],
    augment              = True,
)

val_dataset = ClothingControlNetDataset(
    CONFIG['val_sketch_dir'], CONFIG['val_gt_dir'],
    tokenizer,
    labeler              = None,   # use fallback for val — consistent
    fallback_prompt      = CONFIG['fallback_prompt'],
    img_size             = CONFIG['image_size'],
    mode_probs           = [0.0, 1.0, 0.0],  # always sparse for val
    max_strokes_range    = CONFIG['max_strokes_range'],
    stroke_radius_range  = CONFIG['stroke_radius_range'],
    n_propagated_regions = CONFIG['n_propagated_regions'],
    aug_prob             = 0.0,
    augment              = False,
)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'],
    shuffle=True, num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=4,
    shuffle=False, num_workers=2
)

print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')
print(f'Steps/epoch: {len(train_loader):,}')

## Optimizer + Scheduler + Logging

In [ ]:
# Separate param groups for ControlNet and LoRA (different LRs)
param_groups = [
    {'params': controlnet.parameters(), 'lr': CONFIG['lr_controlnet']},
    {'params': [p for p in unet.parameters() if p.requires_grad],
     'lr': CONFIG['lr_lora']},
]

if CONFIG.get('use_8bit_adam', True):
    try:
        import bitsandbytes as bnb
        optimizer = bnb.optim.AdamW8bit(
            param_groups,
            betas=(0.9, 0.999),
            weight_decay=1e-2,
        )
        print('Using bitsandbytes AdamW8bit optimizer.')
    except Exception as e:
        print(f'bitsandbytes AdamW8bit unavailable ({e}); falling back to torch AdamW.')
        optimizer = torch.optim.AdamW(
            param_groups,
            betas=(0.9, 0.999),
            weight_decay=1e-2,
            foreach=False,
        )
else:
    optimizer = torch.optim.AdamW(
        param_groups,
        betas=(0.9, 0.999),
        weight_decay=1e-2,
        foreach=False,
    )

lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = CONFIG['warmup_steps'],
    num_training_steps = CONFIG['max_train_steps'],
)

scaler = GradScaler()

# ── Logging ────────────────────────────────────────────────────────────────
_LOG_CSV  = os.path.join(CONFIG['logs_dir'], 'training_log.csv')
_LOG_JSON = os.path.join(CONFIG['logs_dir'], 'training_log.json')
_all_logs = []

def log_step(step, loss, lr_cn, lr_lora):
    row = {'step': step, 'loss': round(loss, 6),
           'lr_controlnet': f'{lr_cn:.2e}', 'lr_lora': f'{lr_lora:.2e}'}
    _all_logs.append(row)
    write_header = not os.path.exists(_LOG_CSV)
    with open(_LOG_CSV, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    if step % 200 == 0:
        with open(_LOG_JSON, 'w') as f:
            json.dump(_all_logs, f, indent=2)

print('Optimizer, scheduler, and logging ready.')

## Checkpoint Utilities

In [ ]:
def save_checkpoint(step, loss):
    """
    Saves ControlNet weights + LoRA weights + optimizer state.
    Checkpoint strategy: every 2000 steps.
    Keeps last 3 checkpoints to save disk space.
    """
    ckpt_dir = os.path.join(CONFIG['output_dir'], f'checkpoint-{step}')
    os.makedirs(ckpt_dir, exist_ok=True)

    # ControlNet weights
    controlnet.save_pretrained(os.path.join(ckpt_dir, 'controlnet'))

    # LoRA weights
    unet.save_pretrained(os.path.join(ckpt_dir, 'unet_lora'))

    # Optimizer + scheduler state
    torch.save({
        'step':      step,
        'optimizer': optimizer.state_dict(),
        'scheduler': lr_scheduler.state_dict(),
        'scaler':    scaler.state_dict(),
        'loss':      loss,
    }, os.path.join(ckpt_dir, 'training_state.pt'))

    # Track latest
    with open(os.path.join(CONFIG['output_dir'], 'latest.txt'), 'w') as f:
        f.write(str(step))

    # Keep only last 3 checkpoints
    all_ckpts = sorted([
        d for d in Path(CONFIG['output_dir']).iterdir()
        if d.is_dir() and d.name.startswith('checkpoint-')
    ], key=lambda d: int(d.name.split('-')[1]))
    for old_ckpt in all_ckpts[:-3]:
        shutil.rmtree(old_ckpt)
        print(f'  Removed old checkpoint: {old_ckpt.name}')

    print(f'  [Checkpoint saved] step {step} | loss {loss:.4f}')


def maybe_resume():
    latest_path = os.path.join(CONFIG['output_dir'], 'latest.txt')
    if not os.path.exists(latest_path):
        print('Starting fresh.')
        return 0

    with open(latest_path) as f:
        step = int(f.read().strip())

    ckpt_dir = os.path.join(CONFIG['output_dir'], f'checkpoint-{step}')

    # Load ControlNet
    loaded_cn = ControlNetModel.from_pretrained(os.path.join(ckpt_dir, 'controlnet'))
    controlnet.load_state_dict(loaded_cn.state_dict())
    del loaded_cn

    # Load LoRA
    unet.load_pretrained(os.path.join(ckpt_dir, 'unet_lora'))

    # Load optimizer/scheduler
    state = torch.load(
        os.path.join(ckpt_dir, 'training_state.pt'), map_location=device)
    optimizer.load_state_dict(state['optimizer'])
    lr_scheduler.load_state_dict(state['scheduler'])
    scaler.load_state_dict(state['scaler'])

    # Restore logs
    global _all_logs
    if os.path.exists(_LOG_JSON):
        with open(_LOG_JSON) as f:
            _all_logs = json.load(f)

    torch.cuda.empty_cache()
    print(f'Resumed from step {step}')
    return step


print('Checkpoint utilities ready.')

## Sample Visualization

In [ ]:
@torch.no_grad()
def save_samples(step, fixed_batch):
    """
    4-column grid: Sketch | Color Hint | Generated | Ground Truth
    Runs inference using current controlnet + lora weights.
    """
    controlnet.eval()
    unet.eval()

    pipe = StableDiffusionControlNetPipeline(
        vae          = vae,
        text_encoder = text_encoder,
        tokenizer    = tokenizer,
        unet         = unet,
        controlnet   = controlnet,
        scheduler    = UniPCMultistepScheduler.from_config(
            noise_scheduler.config),
        safety_checker   = None,
        feature_extractor= None,
    )
    pipe = pipe.to(device)

    n   = min(4, fixed_batch['condition'].shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i in range(n):
        cond    = fixed_batch['condition'][i]   # (7, H, W) in [-1, 1]
        gt      = fixed_batch['pixel_values'][i]

        sketch_np = ((cond[:3].permute(1,2,0).cpu().numpy()+1)/2).clip(0,1)
        color_np  = ((cond[3:6].permute(1,2,0).cpu().numpy()+1)/2).clip(0,1)
        gt_np     = ((gt.permute(1,2,0).cpu().numpy()+1)/2).clip(0,1)

        # Build PIL image for the pipe from sketch channel
        sketch_pil = Image.fromarray((sketch_np*255).astype(np.uint8))

        # Run inference — pipe uses our modified controlnet internally
        # We pass the full 7ch condition via controlnet_cond directly
        cond_tensor = cond.unsqueeze(0).to(device, dtype=torch.float16)

        result = pipe(
            prompt                       = CONFIG['fallback_prompt'],
            negative_prompt              = CONFIG['fallback_prompt'].replace(
                'high quality', 'blurry, low quality, deformed, person, mannequin'),
            image                        = sketch_pil,
            num_inference_steps          = 20,
            guidance_scale               = 7.5,
            controlnet_conditioning_scale= 1.0,
            generator                    = torch.Generator(device).manual_seed(CONFIG['seed'])
        ).images[0]
        gen_np = np.array(result) / 255.0

        axes[i,0].imshow(sketch_np[:,:,0], cmap='gray')
        axes[i,0].set_title('Sketch');      axes[i,0].axis('off')
        axes[i,1].imshow(color_np)
        axes[i,1].set_title('Color Hint');  axes[i,1].axis('off')
        axes[i,2].imshow(gen_np)
        axes[i,2].set_title('Generated');   axes[i,2].axis('off')
        axes[i,3].imshow(gt_np)
        axes[i,3].set_title('Ground Truth');axes[i,3].axis('off')

    plt.suptitle(f'Step {step}', fontsize=13)
    plt.tight_layout()
    out_path = os.path.join(CONFIG['samples_dir'], f'step_{step:06d}.png')
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

    del pipe
    torch.cuda.empty_cache()
    controlnet.train()
    unet.train()
    print(f'  [Samples saved] step {step} → {out_path}')


print('Visualization ready.')

## Training Loop

**Checkpoint schedule:** every 2000 steps

**Sample schedule:** every 500 steps

**What to watch:**
- Step 500: sketch structure should be vaguely recognizable
- Step 2000-5000: clear garment shape, rough color
- Step 8000-15000: realistic texture, color hints influencing output

**Loss target:** starts ~0.08-0.12, should drop to ~0.03-0.06 within 5000 steps

In [ ]:
def train():
    global_step = maybe_resume()

    # Fixed val batch — seeded so same samples every run
    _rng_state = np.random.get_state()
    np.random.seed(CONFIG['seed'])
    torch.manual_seed(CONFIG['seed'])
    fixed_batch = next(iter(DataLoader(
        val_dataset, batch_size=4, shuffle=False, num_workers=0,
        worker_init_fn=lambda _: np.random.seed(CONFIG['seed'])
    )))
    np.random.set_state(_rng_state)
    torch.seed()

    # Infinite dataloader
    def cycle(loader):
        while True:
            for b in loader:
                yield b
    data_iter = cycle(train_loader)

    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    n_loss_steps = 0

    progress_bar = tqdm(
        range(CONFIG['max_train_steps']),
        initial=global_step,
        desc='Training', ncols=120
    )

    for step in progress_bar:
        if step < global_step:
            next(data_iter)  # skip already-trained batches
            continue

        batch = next(data_iter)
        condition    = batch['condition'].to(device)     # (B, 7, H, W)
        pixel_values = batch['pixel_values'].to(device)  # (B, 3, H, W)
        input_ids    = batch['input_ids'].to(device)

        # Frozen models do not need activation graphs; keeping them out of the
        # training graph saves enough memory to avoid OOM on 15GB GPUs.
        with torch.no_grad(), autocast():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
            encoder_hidden_states = text_encoder(input_ids)[0]

        with autocast():
            # 1. Add noise
            noise     = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Gradient checkpointing needs at least one floating input with
            # requires_grad=True, otherwise checkpointed modules can skip
            # parameter gradients and AMP has no inf checks to record.
            noisy_latents.requires_grad_(True)
            condition = condition.detach().requires_grad_(True)

            # 2. ControlNet forward — 7ch condition
            down_block_res, mid_block_res = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states = encoder_hidden_states,
                controlnet_cond       = condition,
                return_dict           = False,
            )

            # 5. UNet (with LoRA) forward
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states              = encoder_hidden_states,
                down_block_additional_residuals    = down_block_res,
                mid_block_additional_residual      = mid_block_res,
            ).sample

            # 6. Diffusion loss
            loss = F.mse_loss(noise_pred.float(), noise.float())
            loss = loss / CONFIG['grad_accum_steps']

        scaler.scale(loss).backward()
        running_loss += loss.item() * CONFIG['grad_accum_steps']
        n_loss_steps += 1

        # Optimizer step
        if (step + 1) % CONFIG['grad_accum_steps'] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(controlnet.parameters()) +
                [p for p in unet.parameters() if p.requires_grad],
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        lrs     = lr_scheduler.get_last_lr()
        lr_cn   = lrs[0] if len(lrs) > 0 else CONFIG['lr_controlnet']
        lr_lora = lrs[1] if len(lrs) > 1 else CONFIG['lr_lora']
        avg_loss = running_loss / n_loss_steps

        progress_bar.set_postfix({
            'loss': f"{loss.item()*CONFIG['grad_accum_steps']:.4f}",
            'avg':  f'{avg_loss:.4f}',
            'lr':   f'{lr_cn:.1e}'
        })

        log_step(step, loss.item() * CONFIG['grad_accum_steps'], lr_cn, lr_lora)

        # Visual samples
        if step % CONFIG['sample_every_steps'] == 0:
            save_samples(step, fixed_batch)

        # Checkpoint
        if step % CONFIG['save_every_steps'] == 0 and step > 0:
            save_checkpoint(step, avg_loss)

    # Final save
    save_checkpoint(CONFIG['max_train_steps'], avg_loss)
    controlnet.save_pretrained(os.path.join(CONFIG['output_dir'], 'final_controlnet'))
    unet.save_pretrained(os.path.join(CONFIG['output_dir'], 'final_unet_lora'))
    print('Training complete.')


train()

## Plot Loss Curve

In [ ]:
if os.path.exists(_LOG_JSON):
    with open(_LOG_JSON) as f:
        logs = json.load(f)

    steps  = [r['step'] for r in logs]
    losses = [r['loss'] for r in logs]
    window = 100
    smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')

    plt.figure(figsize=(14, 4))
    plt.plot(steps, losses, alpha=0.15, color='steelblue')
    plt.plot(steps[window-1:], smoothed, color='steelblue',
             linewidth=2, label=f'{window}-step avg')
    plt.xlabel('Step'); plt.ylabel('Diffusion Loss (MSE)')
    plt.title('ControlNet Training Loss')
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['logs_dir'], 'loss_curve.png'), dpi=100)
    plt.show()
    print(f'Final avg loss: {np.mean(losses[-500:]):.4f}')

## Backup Checkpoints (run before session ends)
Zips all checkpoints to `/kaggle/working/` so they're preserved across sessions.

In [ ]:
# Run this manually before your Kaggle session expires
backup_path = '/kaggle/working/controlnet_backup'
shutil.make_archive(backup_path, 'zip', CONFIG['output_dir'])
print(f'Backed up to {backup_path}.zip')
print(f'Size: {os.path.getsize(backup_path+".zip")/1e6:.1f} MB')